## DATASET LOADING AND PERPROCESSING

In [ ]:

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import numpy as np
import string
import requests
import os


In [ ]:

def load_dataset(url, filename):
    if not os.path.exists(filename):
        response = requests.get(url)
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(response.text)
    with open(filename, 'r', encoding='utf-8') as f:
        text = f.read()
    return text

dataset_url = "https://www.gutenberg.org/files/100/100-0.txt"
dataset_file = "shakespeare.txt"

text = load_dataset(dataset_url, dataset_file)

text = text.lower()
text = text.translate(str.maketrans('', '', string.punctuation))


In [ ]:

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

max_sequence_len = 50
input_sequences = np.array(
    pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre')
)

X = input_sequences[:, :-1]
y = input_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)


## Model Design

In [ ]:

model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(128, return_sequences=True))
model.add(LSTM(128))
model.add(Dense(total_words, activation='softmax'))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()


## MODEL TRAINING 

In [ ]:

split_index = int(0.8 * len(X))

X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_model.h5',
    save_best_only=True,
    monitor='val_loss'
)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, checkpoint]
)


## Text Generation

In [ ]:

def generate_text(seed_text, next_words=100, temperature=1.0):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_len-1,
            padding='pre'
        )

        predicted = model.predict(token_list, verbose=0)[0]

        predicted = np.log(predicted + 1e-9) / temperature
        predicted = np.exp(predicted) / np.sum(np.exp(predicted))

        predicted_word_index = np.random.choice(len(predicted), p=predicted)
        output_word = tokenizer.index_word.get(predicted_word_index, "<OOV>")

        seed_text += " " + output_word

    return seed_text


seeds = [
    "to be or not to be",
    "all the world's a stage",
    "romeo romeo wherefore art thou"
]

for seed in seeds:
    generated = generate_text(seed, next_words=50)
    print(f"Seed: {seed}\nGenerated: {generated}\n")
